# 05 · Model comparison on the validation split

> Educational decision-support prototype trained on synthetic PaySim data. Outputs are risk scores and review priorities that help human investigators decide what to review first. This system makes no fraud or AML determination and performs no automatic blocking, account closure, customer risk rating, or regulatory reporting. Results on synthetic data do not establish real-world detection effectiveness, fairness, or regulatory suitability.

Displays validation results produced by `python -m aml_triage train ... --split val` and `python -m aml_triage compare --split val`. **No test-split numbers appear here**: the test split stays locked until the operating point is frozen (Milestone 6). The human discussion lives in `reports/model_comparison_narrative.md`.

In [ ]:
from pathlib import Path

import json
import pandas as pd
from IPython.display import Markdown, display

from aml_triage.config import load

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = load(ROOT / "configs" / "base.yaml")
REPORTS = ROOT / cfg.paths.reports_dir
RUNS = ROOT / cfg.paths.models_dir / "runs"
print("config hash:", cfg.config_hash())

## Runs and fit scope

Every estimator must have been fitted on `train` only and scored on `val` only.

In [ ]:
rows = []
for d in sorted(RUNS.glob("*/val_metrics.json")) if RUNS.exists() else []:
    m = json.loads(d.read_text())
    K = str(cfg.review.primary_k)
    rows.append({"run": d.parent.name, "fitted_on": m["fit_scope"]["fitted_on"], "scored": m["fit_scope"]["transformed_on"], "fit_s": m["fit_seconds"], "PR-AUC": round(m["metrics"]["pr_auc"], 4), f"Recall@{K}": m["recall_at_k"][K]["mean_over_periods"], "degenerate": m["metrics"]["degenerate_scores"]})
display(pd.DataFrame(rows) if rows else Markdown("_Run `python -m aml_triage train --split val` first._"))

## Comparison report (validation sections only are populated at this stage)

In [ ]:
p = REPORTS / "model_comparison.md"
display(Markdown(p.read_text().replace("](figures/", f"]({REPORTS.as_posix()}/figures/"))) if p.exists() else Markdown("_Run `python -m aml_triage compare --split val` first._"))